In [1]:
import torch
import sys
import platform

print("=== IndustrialDefect AI — GPU Check ===")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Platform:", platform.system())

print("\n=== GPU Information ===")

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU detected!")

=== IndustrialDefect AI — GPU Check ===
Python: 3.13.16
PyTorch: 2.11.0+cu130
Platform: Linux

=== GPU Information ===
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [2]:
import torch
import time

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device selected:", device)

# Création de deux matrices
A = torch.randn(1000, 1000, device=device)
B = torch.randn(1000, 1000, device=device)

# Multiplication matricielle
start = time.perf_counter()

C = torch.matmul(A, B)

if device.type == "cuda":
    torch.cuda.synchronize()

end = time.perf_counter()

print("Result shape:", C.shape)
print("Execution device:", C.device)
print("Execution time:", round(end - start, 4), "seconds")

Device selected: cuda
Result shape: torch.Size([1000, 1000])
Execution device: cuda:0
Execution time: 0.1119 seconds


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# Reproductibilité
torch.manual_seed(42)

# Sélection du GPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Architecture du modèle
def build_model():
    return nn.Sequential(
        nn.Linear(4, 16),
        nn.ReLU(),
        nn.Linear(16, 2)
    )

model = build_model().to(device)

# Données synthétiques
X = torch.randn(256, 4, device=device)

y = (X[:, 0] + X[:, 1] > 0).long()

# Fonction de perte et optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Entraînement
model.train()

for epoch in range(40):

    optimizer.zero_grad()

    predictions = model(X)

    loss = criterion(predictions, y)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/40 "
            f"- Loss: {loss.item():.4f}"
        )

model.eval()

print("Training completed!")

Device: cuda
Epoch 10/40 - Loss: 0.4326
Epoch 20/40 - Loss: 0.2659
Epoch 30/40 - Loss: 0.1573
Epoch 40/40 - Loss: 0.1048
Training completed!


In [4]:
from pathlib import Path
import json

# Répertoire persistant lors de la sauvegarde d'une version
OUTPUT_DIR = Path("/kaggle/working/infra_artifacts")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chemins des artefacts
model_path = OUTPUT_DIR / "tiny_model_state_dict.pt"
metadata_path = OUTPUT_DIR / "experiment_info.json"

# 1. Sauvegarder les poids
torch.save(model.state_dict(), model_path)

# 2. Créer une entrée de référence
reference_input = torch.tensor(
    [[0.2, -0.1, 0.4, 0.7]],
    device=device
)

with torch.no_grad():
    reference_output = model(reference_input).cpu()

# 3. Sauvegarder les métadonnées
metadata = {
    "project": "IndustrialDefect AI",
    "experiment": "INFRA-07",
    "model": "Tiny Neural Network",
    "framework": "PyTorch",
    "torch_version": torch.__version__,
    "device": str(device),
    "epochs": 40,
    "learning_rate": 0.01,
    "random_seed": 42,
    "reference_input": reference_input.cpu().tolist(),
    "reference_output": reference_output.tolist()
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

# Vérifier les artefacts
print("=== Saved Artifacts ===")

for file in OUTPUT_DIR.iterdir():
    print(f"{file.name} - {file.stat().st_size} bytes")

print("\nModel saved successfully!")

=== Saved Artifacts ===
tiny_model_state_dict.pt - 3113 bytes
experiment_info.json - 561 bytes

Model saved successfully!


In [5]:
# Créer une nouvelle instance du modèle
restored_model = build_model()

# Charger les poids sauvegardés
state_dict = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

restored_model.load_state_dict(state_dict)
restored_model.eval()

# Recharger les données de référence
with open(metadata_path, "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

test_input = torch.tensor(
    saved_metadata["reference_input"],
    dtype=torch.float32
)

expected_output = torch.tensor(
    saved_metadata["reference_output"],
    dtype=torch.float32
)

# Calculer la prédiction du modèle restauré
with torch.inference_mode():
    restored_output = restored_model(test_input)

# Comparer les résultats
is_valid = torch.allclose(
    restored_output,
    expected_output,
    atol=1e-5,
    rtol=1e-5
)

print("Original prediction:", expected_output)
print("Restored prediction:", restored_output)
print("Predictions match:", is_valid)

assert is_valid, "Model restoration failed!"

print("\nModel restoration successful!")

Original prediction: tensor([[-0.4784,  0.0699]])
Restored prediction: tensor([[-0.4784,  0.0699]])
Predictions match: True

Model restoration successful!
